# Consciencia para IA — Experimento 01

Clona el repo y ejecuta la línea base de continuidad, dependencia de trayectoria, X y atractor.


In [ ]:
!rm -rf /content/Consciencia-Skill
!git clone -q https://github.com/chrishotza/Consciencia-Skill.git /content/Consciencia-Skill
%pip -q install -r /content/Consciencia-Skill/requirements.txt
import sys
sys.path.insert(0, "/content/Consciencia-Skill")
import numpy as np
import pandas as pd
from src.ontto.dynamics import Config, simulate, common_attractor_simulation, metrics
SEED = 42
N = 6000
WARMUP = 500
rng = np.random.default_rng(SEED)


In [ ]:
u = np.empty(N)
state = 1.0
for t in range(N):
    if rng.random() < (0.03 if t < N//2 else 0.11):
        state *= -1.0
    u[t] = state

full = simulate(u, Config(), seed=SEED)
ablated = simulate(
    u,
    Config(use_memory=False, use_pressure=False, use_cross=False, use_x=False, use_attractor=False),
    seed=SEED,
)
print("FULL:", metrics(full, WARMUP))
print("ABLATION:", metrics(ablated, WARMUP))


In [ ]:
prefix_a = np.ones(400)
prefix_b = -np.ones(400)
suffix = np.ones(600)
cfg = Config(beta_memory=0.97)
a = simulate(np.r_[prefix_a, suffix], cfg, seed=SEED)
b = simulate(np.r_[prefix_b, suffix], cfg, seed=SEED)
gap = np.abs(a["state"][400:] - b["state"][400:])
print("PATH_DEPENDENCE_MEAN_GAP:", float(gap.mean()))
print("PATH_DEPENDENCE_FINAL_GAP:", float(gap[-1]))


In [ ]:
q = full["q"][WARMUP:]
cross = full["cross"][WARMUP:]
hi = cross >= np.quantile(cross, 0.75)
lo = cross <= np.quantile(cross, 0.25)
x_high = float(np.mean((q == "X")[hi]))
x_low = float(np.mean((q == "X")[lo]))
print("X_OCCUPANCY:", float(np.mean(q == "X")))
print("X_GIVEN_HIGH_CROSS:", x_high)
print("X_GIVEN_LOW_CROSS:", x_low)
print("CRITICAL_LIFT:", x_high - x_low)


In [ ]:
inputs = rng.choice([-1.0, 1.0], size=(8, N))
coupled = common_attractor_simulation(inputs, n_agents=8, cfg=Config(coupling_gain=0.4), seed=SEED)
uncoupled = common_attractor_simulation(inputs, n_agents=8, cfg=Config(coupling_gain=0.0), seed=SEED)
spread_c = float(np.mean(np.std(coupled["states"][:, -500:], axis=0)))
spread_u = float(np.mean(np.std(uncoupled["states"][:, -500:], axis=0)))
print("ATTRACTOR_SPREAD_COUPLED:", spread_c)
print("ATTRACTOR_SPREAD_UNCOUPLED:", spread_u)
print("ATTRACTOR_SPREAD_REDUCTION:", 1.0 - spread_c / spread_u)


## Interpretación

Repetir con múltiples semillas antes de tratar una señal como robusta.
